# 18 · 循环神经网络与 BPTT

> **本节属于 Part 7 · 序列模型 RNN/LSTM。**

前面处理的都是固定大小的输入（向量、图像）。但语言、语音、时间序列都是**变长序列**。**循环神经网络 (RNN)** 通过一个随时间更新的"隐藏状态"来处理序列：

$$h_t = \tanh(x_t W_{xh} + h_{t-1} W_{hh} + b)$$

本节实现 RNN，理解它的反向传播——**随时间反向传播 (BPTT)**——并惊喜地发现：我们的 autograd 引擎能**自动**完成 BPTT。我们还会亲眼看到 vanilla RNN 的致命弱点：**梯度消失**。

## 学习目标

- 实现 `RNNCell` 与在序列上展开的 `RNN`
- 理解 **BPTT**：序列在时间上展开成一张深图，autograd 自动沿它反传
- **可视化梯度消失**：长序列里早期时间步收到的梯度指数级衰减
- 在一个"记忆任务"上训练 RNN

## 实现与 BPTT

把序列"按时间步展开"：第 $t$ 步用上一步的隐藏态 $h_{t-1}$。展开后就是一张很深的计算图，反向传播沿着它从最后一步回到第一步——这就是 BPTT。**我们一行 BPTT 都不用写**，autograd 全包了。

In [ ]:
import inspect
import numpy as np
import matplotlib.pyplot as plt
import minitorch
from minitorch import Tensor, nn, no_grad
from minitorch.optim import Adam

print(inspect.getsource(nn.RNN.forward))

## 梯度消失：vanilla RNN 的硬伤

把一个长序列喂进 RNN，对最后的隐藏态求和并反向，然后看**每个时间步的输入收到的梯度大小**。你会看到：越靠前的时间步，梯度越小——这就是**梯度消失**，它让 RNN 难以学习长距离依赖。

In [ ]:
minitorch.set_seed(0)
rnn = nn.RNN(1, 32)
T = 60
x = Tensor(np.random.randn(1, T, 1))
_, h_N = rnn(x)
h_N.sum().backward()

grad_norm = np.abs(x.grad[0, :, 0])          # 每个时间步输入梯度的大小
plt.figure(figsize=(6, 3.2))
plt.plot(range(T), grad_norm, marker=".")
plt.yscale("log"); plt.xlabel("time step t"); plt.ylabel("|grad of input_t| (log)")
plt.title("Vanishing gradient: early steps get tiny gradients")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f"最后一步梯度 / 第一步梯度 ≈ {grad_norm[-1]/max(grad_norm[0],1e-12):.1e} 倍")

## 一个"记忆任务"

任务：序列的**第一个** token 是信号（+1 或 −1），后面全是噪声。模型要在读完整条序列后，回答"第一个 token 是正还是负"。这要求把第一步的信息**一路记到最后**。

In [ ]:
def make_memory(n, T, seed):
    rng = np.random.RandomState(seed)
    sig = rng.choice([-1.0, 1.0], size=(n, 1))
    noise = rng.randn(n, T - 1) * 0.5
    X = np.concatenate([sig, noise], axis=1)[:, :, None]   # (n, T, 1)
    y = (sig[:, 0] > 0).astype(int)
    return X, y

def train_rnn(T, epochs=80):
    minitorch.set_seed(0)
    rnn = nn.RNN(1, 32); head = nn.Linear(32, 2)
    opt = Adam(rnn.parameters() + head.parameters(), lr=5e-3)
    loss_fn = nn.CrossEntropyLoss()
    Xtr, ytr = make_memory(400, T, 0)
    for ep in range(epochs):
        opt.zero_grad()
        _, h_N = rnn(Tensor(Xtr))
        loss_fn(head(h_N), ytr).backward()
        opt.step()
    Xte, yte = make_memory(400, T, 1)
    _, h_N = rnn(Tensor(Xte))
    with no_grad():
        acc = (head(h_N).data.argmax(1) == yte).mean()
    return acc

print(f"短序列 T=15： RNN 测试准确率 = {train_rnn(15)*100:.1f}%   （能学会）")
print(f"长序列 T=50： RNN 测试准确率 = {train_rnn(50)*100:.1f}%   （梯度消失，退化到随机猜）")

短序列上 RNN 轻松学会；但序列一长，由于梯度消失，RNN 几乎退化到 50%（随机猜测）。这正是下一节 **LSTM** 要解决的问题。

## PyTorch 对照

`nn.RNN` 的概念完全一致（PyTorch 默认一次返回整段输出与最终隐藏态）。

In [ ]:
import torch
trnn = torch.nn.RNN(input_size=1, hidden_size=8, batch_first=True)
x = torch.randn(3, 6, 1)
out, h_n = trnn(x)
print("PyTorch RNN 输出形状:", tuple(out.shape), " 最终隐藏态:", tuple(h_n.shape))
print("minitorch RNN：outputs 列表(每步 (N,H)) + 最终 h_N，二者概念一致")

## 📦 沉淀进 minitorch

`RNNCell / RNN` 在 `minitorch/nn/rnn.py`，BPTT 梯度由 `tests/test_rnn.py` 的数值梯度检查守护。**关键收获**：序列模型不需要特殊的反向算法——把每步前向写出来，autograd 自动完成 BPTT。

## 小练习

1. **隐藏维度**：把 `hidden_size` 从 32 调到 8 或 128，对短/长序列任务的影响如何？
2. **梯度爆炸**：把 `RNNCell` 的 `tanh` 换成不饱和的恒等映射，重看梯度大小——可能从"消失"变成"爆炸"。
3. **梯度裁剪**：实现简单的梯度裁剪（把过大的梯度按范数缩放），它能缓解爆炸但治不了消失，想想为什么。

## 小结 & 下一站

✅ 我们实现了 RNN，确认 autograd 自动完成 BPTT，并**可视化了梯度消失**——看到了 vanilla RNN 学不动长依赖的根本原因。

**下一站 → `19_lstm_and_sequence_task`**：实现 **LSTM**。它用"门控 + 记忆细胞"打通了一条让梯度顺畅流动的"高速公路"，从而在长序列任务上**完胜** vanilla RNN。